In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from collections import Counter

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.dpi': 120
})

# --- Shared configuration ---
N_DAYS = 300
N_SITES = 50
RECORDS_PER_SITE_PER_DAY = 4
FAILURE_START = 150
FAILURE_END = 300
QUARTERLY_REVIEW_DAYS = [90, 180, 270]
WINDOW = 7

AE_CATEGORIES = [
    'Headache', 'Nausea', 'Fatigue', 'Dizziness', 'Injection site pain',
    'Diarrhea', 'Insomnia', 'Arthralgia', 'Cough', 'Rash',
    'Back pain', 'Vomiting', 'Pyrexia', 'Hypertension', 'Upper resp infection',
    'Peripheral edema', 'ALT increased', 'Cardiac event', 'Hepatotoxicity', 'Anaphylaxis'
]

TRUE_PROBS = np.array([
    0.15, 0.12, 0.10, 0.08, 0.08,
    0.07, 0.06, 0.05, 0.05, 0.04,
    0.04, 0.03, 0.03, 0.025, 0.02,
    0.015, 0.01, 0.008, 0.005, 0.002
])
TRUE_PROBS = TRUE_PROBS / TRUE_PROBS.sum()

SUPPRESSED = [16, 17, 18]  # ALT increased, Cardiac event, Hepatotoxicity
REDIRECT_TO = [0, 2]       # Headache, Fatigue


def compute_shannon_entropy(records, n_categories=20):
    if len(records) == 0:
        return 0.0
    counts = Counter(records)
    total = sum(counts.values())
    probs = np.array([counts.get(i, 0) / total for i in range(n_categories)])
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))


def compute_category_frequencies(records, n_categories=20):
    if len(records) == 0:
        return np.zeros(n_categories)
    counts = Counter(records)
    total = sum(counts.values())
    return np.array([counts.get(i, 0) / total for i in range(n_categories)])


def entropy_contributions(freqs):
    c = np.zeros_like(freqs)
    mask = freqs > 0
    c[mask] = -freqs[mask] * np.log2(freqs[mask])
    return c


print(f"Config: {N_SITES} sites, {N_DAYS} days, ~{N_SITES * RECORDS_PER_SITE_PER_DAY} records/day")
print(f"Failure: Day {FAILURE_START}–{FAILURE_END}, Site 0 miscodes {[AE_CATEGORIES[i] for i in SUPPRESSED]}")
print(f"Redirect to: {[AE_CATEGORIES[i] for i in REDIRECT_TO]}")

---
## Figure 1: The Entropy Budget

**Post section**: *The Entropy Budget of a Process*

**Purpose**: Teach the core concept. Each pipeline stage has an expected entropy cost. Deviations = signals.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

stages = ['Collection\n(Raw Data)', 'Entry &\nValidation', 'Aggregation\n& Analysis', 'Regulatory\nReporting']
stage_x = [0, 1, 2, 3]

expected_h = [3.9, 3.85, 2.1, 2.1]
anomalous_h = [3.9, 3.72, 2.1, 2.1]

ax.plot(stage_x, expected_h, 'o-', color='#2E86AB', linewidth=2.5, markersize=12,
        label='Expected entropy budget', zorder=5)
ax.plot(stage_x, anomalous_h, 's--', color='#E84855', linewidth=2, markersize=10,
        label='Anomalous path (leakage at Entry)', zorder=5)

# Annotate expected transitions
ax.annotate('Small ΔH\n(noise removal)', xy=(0.5, 3.88), fontsize=9,
            ha='center', color='#666666', style='italic')
ax.annotate('Large ΔH\n(intentional aggregation)', xy=(1.5, 3.0), fontsize=9,
            ha='center', color='#666666', style='italic')
ax.annotate('≈ 0 ΔH\n(should preserve)', xy=(2.5, 2.17), fontsize=9,
            ha='center', color='#666666', style='italic')

# Annotate anomaly
ax.annotate('Unexpected\ndrop — investigate', xy=(1, 3.72), xytext=(1.35, 3.48),
            fontsize=10, fontweight='bold', color='#E84855',
            arrowprops=dict(arrowstyle='->', color='#E84855', lw=1.5))

# Fill the leak
ax.fill_between([0.92, 1.08], [3.85, 3.85], [3.72, 3.72], 
                alpha=0.25, color='#E84855', label='Information leaked')

ax.set_xticks(stage_x)
ax.set_xticklabels(stages, fontsize=11)
ax.set_ylabel('Shannon Entropy (bits)', fontsize=11)
ax.set_title('The Entropy Budget: Expected vs. Anomalous Information Flow', fontweight='bold', fontsize=13)
ax.legend(loc='center right', fontsize=9)
ax.set_ylim(1.5, 4.2)
ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.savefig('entropy_budget.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Figure 2: Three Failure Modes, Three Entropy Signatures

**Post section**: *Three Failure Modes, Three Entropy Signatures*

**Purpose**: Teach the taxonomy. Show that different failures produce different, recognizable entropy fingerprints.

In [ ]:
n_cats = 10
cat_labels = [f'Cat {i+1}' for i in range(n_cats)]

healthy = np.array([0.25, 0.18, 0.14, 0.10, 0.08, 0.07, 0.06, 0.05, 0.04, 0.03])
healthy = healthy / healthy.sum()

# Leakage: rare categories suppressed, mass redistributed to common ones
leakage = healthy.copy()
leakage[7:] = 0
leakage[:3] += healthy[7:].sum() / 3
leakage = leakage / leakage.sum()

# Corruption: noise makes distribution more uniform
corruption = 0.6 * healthy + 0.4 * np.ones(n_cats) / n_cats
corruption = corruption / corruption.sum()

# Distortion: shape changes but total entropy is roughly preserved
distortion = healthy.copy()
distortion[0], distortion[4] = distortion[4], distortion[0]
distortion[2], distortion[7] = distortion[7], distortion[2]
distortion = distortion / distortion.sum()

def H(p):
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
x = np.arange(n_cats)
width = 0.35

configs = [
    ('Leakage (Suppression)', leakage,
     f'H: {H(healthy):.2f} → {H(leakage):.2f} bits\n▼ Entropy DROPS', '#E84855'),
    ('Corruption (Noise Injection)', corruption,
     f'H: {H(healthy):.2f} → {H(corruption):.2f} bits\n▲ Entropy RISES', '#E84855'),
    ('Distortion (Shape Shift)', distortion,
     f'H: {H(healthy):.2f} → {H(distortion):.2f} bits\n≈ Entropy STABLE, shape changes', '#FFC107')
]

for ax, (title, after, label, color) in zip(axes, configs):
    ax.bar(x - width/2, healthy, width, label='Before', color='#2E86AB', alpha=0.7)
    ax.bar(x + width/2, after, width, label='After', color='#E84855', alpha=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(cat_labels, rotation=45, ha='right', fontsize=8)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.15, axis='y')
    ax.text(0.5, 0.92, label, transform=ax.transAxes, fontsize=9,
            ha='center', va='top', color=color, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.85))

axes[0].set_ylabel('Relative Frequency')
plt.suptitle('Three Failure Modes, Three Entropy Signatures', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('entropy_three_signatures.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Healthy:     H = {H(healthy):.3f} bits")
print(f"Leakage:     H = {H(leakage):.3f} bits  (ΔH = {H(leakage)-H(healthy):+.3f})")
print(f"Corruption:  H = {H(corruption):.3f} bits  (ΔH = {H(corruption)-H(healthy):+.3f})")
print(f"Distortion:  H = {H(distortion):.3f} bits  (ΔH = {H(distortion)-H(healthy):+.3f})")

---
## Run the Main Simulation

One simulation run feeds Figures 3, 4, and 5. We track both aggregate and per-site data.

In [ ]:
np.random.seed(42)

# --- Aggregate tracking ---
entropy_input, entropy_entered = [], []
input_buffer, entered_buffer = [], []
all_daily_entered = []  # Store for decomposition

# --- Per-site tracking ---
SITE_WINDOW = 14  # Larger window for per-site (fewer records per site per day)
site_buffers = {s: [] for s in range(N_SITES)}
site_entropies = {s: [] for s in range(N_SITES)}

# Pre/during failure record collectors (for decomposition)
pre_failure_entered = []
during_failure_entered = []

for day in range(N_DAYS):
    failure_active = FAILURE_START <= day < FAILURE_END
    day_input, day_entered = [], []
    
    for site in range(N_SITES):
        n_rec = np.random.poisson(RECORDS_PER_SITE_PER_DAY)
        recs = np.random.choice(len(TRUE_PROBS), size=n_rec, p=TRUE_PROBS).tolist()
        day_input.extend(recs)
        
        # Apply failure at site 0
        if failure_active and site == 0:
            entered = []
            for r in recs:
                if r in SUPPRESSED and np.random.random() < 0.7:
                    entered.append(np.random.choice(REDIRECT_TO))
                else:
                    entered.append(r)
        else:
            entered = recs[:]
        
        day_entered.extend(entered)
        
        # Per-site buffer
        site_buffers[site].append(entered)
        if len(site_buffers[site]) > SITE_WINDOW:
            site_buffers[site].pop(0)
        flat_site = [r for d in site_buffers[site] for r in d]
        site_entropies[site].append(compute_shannon_entropy(flat_site))
    
    all_daily_entered.append(day_entered)
    
    if day < FAILURE_START:
        pre_failure_entered.extend(day_entered)
    else:
        during_failure_entered.extend(day_entered)
    
    # Aggregate rolling window
    input_buffer.append(day_input)
    entered_buffer.append(day_entered)
    if len(input_buffer) > WINDOW:
        input_buffer.pop(0)
        entered_buffer.pop(0)
    
    flat_in = [r for d in input_buffer for r in d]
    flat_out = [r for d in entered_buffer for r in d]
    entropy_input.append(compute_shannon_entropy(flat_in))
    entropy_entered.append(compute_shannon_entropy(flat_out))

entropy_input = np.array(entropy_input)
entropy_entered = np.array(entropy_entered)
entropy_delta = entropy_entered - entropy_input
days = np.arange(N_DAYS)

for s in range(N_SITES):
    site_entropies[s] = np.array(site_entropies[s])

# Baseline stats
bl_mean = entropy_delta[:FAILURE_START].mean()
bl_std = entropy_delta[:FAILURE_START].std()
threshold = bl_mean - 2.5 * bl_std

print(f"Simulation complete: {N_DAYS} days, {N_SITES} sites")
print(f"\nBaseline ΔH:  {bl_mean:.6f} ± {bl_std:.6f} bits")
print(f"Failure ΔH:   {entropy_delta[FAILURE_START:].mean():.6f} bits")
print(f"Threshold:    {threshold:.6f} bits")

---
## Figure 3: Main Evidence — Entropy Divergence and Detection Gap

**Post section**: *A Simulation: Entropy Monitoring in a Clinical Data Pipeline*

**Purpose**: Show that entropy monitoring detects the failure weeks/months before a quarterly review.

**Design choice**: The top panel zooms into the ΔH (entropy delta) rather than showing raw entropy at both stages, because the raw entropy lines nearly overlap and the signal is invisible at that scale. The delta isolates the signal. The bottom panel shows the detection timing comparison.

In [ ]:
# Find aggregate detection day
consecutive = 0
entropy_detection_day = None
for d in range(FAILURE_START, N_DAYS):
    if entropy_delta[d] < threshold:
        consecutive += 1
        if consecutive >= 3:
            entropy_detection_day = d - 2
            break
    else:
        consecutive = 0

quarterly_detection = 270

# --- Plot: 2-panel (delta + detection timing) ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7.5), sharex=True,
                                gridspec_kw={'height_ratios': [2.5, 1]})

# Top: Entropy delta with clear annotation
ax1.plot(days, entropy_delta, color='#434343', alpha=0.5, linewidth=0.8)
# Add smoothed trend for readability
smooth_window = 14
smoothed = np.convolve(entropy_delta, np.ones(smooth_window)/smooth_window, mode='same')
ax1.plot(days[smooth_window//2:-smooth_window//2], 
         smoothed[smooth_window//2:-smooth_window//2],
         color='#2E86AB', linewidth=2, label='ΔH (14-day moving average)', zorder=4)

ax1.axhline(0, color='gray', linestyle='-', alpha=0.3)
ax1.axhline(threshold, color='#E84855', linestyle='--', alpha=0.7, linewidth=1.2,
            label=f'Alert threshold (−2.5σ = {threshold:.4f})')
ax1.axhline(bl_mean, color='#999999', linestyle=':', alpha=0.5,
            label=f'Baseline mean ({bl_mean:.5f})')
ax1.axvspan(FAILURE_START, FAILURE_END, alpha=0.07, color='red', label='Failure period')
ax1.axvline(FAILURE_START, color='red', linestyle='--', alpha=0.4, linewidth=0.8)

if entropy_detection_day:
    ax1.axvline(entropy_detection_day, color='#E84855', linestyle='-', alpha=0.5, linewidth=1)
    ax1.annotate(f'Alert: Day {entropy_detection_day}', 
                 xy=(entropy_detection_day, threshold),
                 xytext=(entropy_detection_day + 15, threshold + 0.002),
                 fontsize=9, color='#E84855', fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color='#E84855', lw=1.2))

ax1.set_ylabel('ΔH (entered − input) [bits]')
ax1.set_title('Entropy Delta: Deviation from Expected Information Budget', fontweight='bold')
ax1.legend(loc='lower left', fontsize=9)
ax1.grid(True, alpha=0.15)

# Bottom: Detection timing
ax2.set_xlim(0, N_DAYS)
ax2.set_ylim(-0.5, 1.5)
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['Quarterly\nReview', 'Entropy\nMonitor'], fontsize=10)

for qd in QUARTERLY_REVIEW_DAYS:
    if qd <= FAILURE_START:
        color = '#4CAF50'
    elif qd == 180:
        color = '#FFC107'
    else:
        color = '#E84855'
    ax2.scatter(qd, 0, s=120, color=color, marker='s', zorder=5, edgecolors='black', linewidth=0.5)

if entropy_detection_day:
    ax2.scatter(entropy_detection_day, 1, s=120, color='#E84855', marker='D', 
                zorder=5, edgecolors='black', linewidth=0.5)
    ax2.annotate(f'Day {entropy_detection_day}', (entropy_detection_day, 1),
                 textcoords='offset points', xytext=(10, 5), fontsize=9, color='#E84855')

ax2.annotate(f'Day {quarterly_detection}', (quarterly_detection, 0),
             textcoords='offset points', xytext=(10, -12), fontsize=9, color='#E84855')

if entropy_detection_day:
    gap = quarterly_detection - entropy_detection_day
    mid = (entropy_detection_day + quarterly_detection) / 2
    ax2.annotate('', xy=(quarterly_detection, 0.5), xytext=(entropy_detection_day, 0.5),
                 arrowprops=dict(arrowstyle='<->', color='#2E86AB', lw=1.5))
    ax2.text(mid, 0.68, f'{gap}-day\ndetection gap', ha='center', fontsize=10, 
             color='#2E86AB', fontweight='bold')

ax2.axvspan(FAILURE_START, FAILURE_END, alpha=0.07, color='red')
ax2.axvline(FAILURE_START, color='red', linestyle='--', alpha=0.4, linewidth=0.8)
ax2.set_xlabel('Day')
ax2.set_title('Detection Timing: Entropy Monitor vs. Quarterly Review', fontweight='bold')
ax2.grid(True, alpha=0.15, axis='x')

plt.tight_layout()
plt.savefig('entropy_main.png', dpi=150, bbox_inches='tight')
plt.show()

if entropy_detection_day:
    print(f"\n--- Detection Summary ---")
    print(f"Failure introduced:          Day {FAILURE_START}")
    print(f"Entropy monitor detection:   Day {entropy_detection_day} ({entropy_detection_day - FAILURE_START} days after onset)")
    print(f"Quarterly review detection:  Day {quarterly_detection} ({quarterly_detection - FAILURE_START} days after onset)")
    print(f"Detection advantage:          {gap} days")

---
## Figure 5: Per-Category Entropy Decomposition

**Post section**: Follows the per-site figure.

**Purpose**: Show actionability. Once entropy flags an anomaly, the decomposition tells you *which categories* are driving the shift — giving the auditor an immediate investigative lead.

**Note on noise**: Some uninvolved categories will show non-zero shifts due to random sampling variation. This is realistic — in real data you'd see the same thing. The auditor uses the decomposition as a starting point for investigation, not as a definitive answer.

In [ ]:
freq_pre = compute_category_frequencies(pre_failure_entered)
freq_during = compute_category_frequencies(during_failure_entered)

contrib_pre = entropy_contributions(freq_pre)
contrib_during = entropy_contributions(freq_during)
contrib_delta = contrib_during - contrib_pre

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

x = np.arange(len(AE_CATEGORIES))
width = 0.35

# Top: frequency comparison
ax1.bar(x - width/2, freq_pre, width, label='Pre-failure', color='#2E86AB', alpha=0.7)
ax1.bar(x + width/2, freq_during, width, label='During failure (entered)', color='#E84855', alpha=0.7)
ax1.set_xticks(x)
ax1.set_xticklabels(AE_CATEGORIES, rotation=45, ha='right', fontsize=8)
ax1.set_ylabel('Relative Frequency')
ax1.set_title('AE Category Distribution: Before vs. During Failure', fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.15, axis='y')

for idx in SUPPRESSED:
    ax1.get_xticklabels()[idx].set_color('#E84855')
    ax1.get_xticklabels()[idx].set_fontweight('bold')
for idx in REDIRECT_TO:
    ax1.get_xticklabels()[idx].set_color('#2E86AB')
    ax1.get_xticklabels()[idx].set_fontweight('bold')

# Bottom: entropy contribution delta
colors = ['#E84855' if d < -0.0005 else '#4CAF50' if d > 0.0005 else '#BBBBBB' for d in contrib_delta]
ax2.bar(x, contrib_delta, color=colors, alpha=0.7, edgecolor='white')
ax2.axhline(0, color='black', linewidth=0.5)
ax2.set_xticks(x)
ax2.set_xticklabels(AE_CATEGORIES, rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('ΔEntropy Contribution (bits)')
ax2.set_title('Per-Category Entropy Change: Where Is Information Being Lost?', fontweight='bold')
ax2.grid(True, alpha=0.15, axis='y')

for idx in SUPPRESSED:
    ax2.get_xticklabels()[idx].set_color('#E84855')
    ax2.get_xticklabels()[idx].set_fontweight('bold')
for idx in REDIRECT_TO:
    ax2.get_xticklabels()[idx].set_color('#2E86AB')
    ax2.get_xticklabels()[idx].set_fontweight('bold')

plt.tight_layout()
plt.savefig('entropy_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

print("Red labels: suppressed AEs | Blue labels: redirect targets")
print("\nTop 5 categories by |ΔEntropy Contribution|:")
sorted_idx = np.argsort(np.abs(contrib_delta))[::-1]
for rank, i in enumerate(sorted_idx[:5], 1):
    involved = '← SUPPRESSED' if i in SUPPRESSED else '← REDIRECT TARGET' if i in REDIRECT_TO else '(sampling noise)'
    print(f"  {rank}. {AE_CATEGORIES[i]:25s}  Δ = {contrib_delta[i]:+.5f} bits  {involved}")

---
## Figure 6: Sensitivity Analysis

**Post section**: Honest limitations.

**Purpose**: Show where entropy monitoring works well and where it struggles. Vary suppression rate from 10% to 90%, run 20 Monte Carlo trials each.

In [ ]:
def run_trial(suppression_rate, seed, n_failing_sites=1):
    """Run one simulation trial and return days-after-onset to detection (or None)."""
    np.random.seed(seed)
    in_buf, out_buf = [], []
    e_delta = []
    
    for day in range(N_DAYS):
        failure_active = day >= FAILURE_START
        all_in, all_out = [], []
        
        for site in range(N_SITES):
            n_rec = np.random.poisson(RECORDS_PER_SITE_PER_DAY)
            recs = np.random.choice(len(TRUE_PROBS), size=n_rec, p=TRUE_PROBS).tolist()
            all_in.extend(recs)
            
            if failure_active and site < n_failing_sites:
                entered = []
                for r in recs:
                    if r in SUPPRESSED and np.random.random() < suppression_rate:
                        entered.append(np.random.choice(REDIRECT_TO))
                    else:
                        entered.append(r)
                all_out.extend(entered)
            else:
                all_out.extend(recs)
        
        in_buf.append(all_in); out_buf.append(all_out)
        if len(in_buf) > WINDOW: in_buf.pop(0); out_buf.pop(0)
        
        fi = [r for d in in_buf for r in d]
        fo = [r for d in out_buf for r in d]
        e_delta.append(compute_shannon_entropy(fo) - compute_shannon_entropy(fi))
    
    e_delta = np.array(e_delta)
    mu = e_delta[:FAILURE_START].mean()
    sigma = e_delta[:FAILURE_START].std()
    thresh = mu - 2.5 * sigma
    
    c = 0
    for d in range(FAILURE_START, N_DAYS):
        if e_delta[d] < thresh:
            c += 1
            if c >= 3: return d - 2 - FAILURE_START
        else:
            c = 0
    return None


suppression_rates = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
n_trials = 20
results = {}

for sr in suppression_rates:
    detections = [run_trial(sr, seed=1000+t) for t in range(n_trials)]
    detected = [d for d in detections if d is not None]
    results[sr] = {
        'rate': len(detected) / n_trials,
        'mean': np.mean(detected) if detected else None,
        'std': np.std(detected) if detected else None
    }
    det_str = f"mean {results[sr]['mean']:.0f} ± {results[sr]['std']:.0f} days" if detected else "—"
    print(f"Suppression {sr:.0%}: {len(detected)}/{n_trials} detected ({results[sr]['rate']:.0%}), {det_str}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: Detection rate
rates = [results[sr]['rate'] for sr in suppression_rates]
bar_colors = ['#E84855' if r < 0.7 else '#FFC107' if r < 0.95 else '#2E86AB' for r in rates]
ax1.bar(range(len(suppression_rates)), rates, color=bar_colors, alpha=0.75, edgecolor='white')
ax1.set_xticks(range(len(suppression_rates)))
ax1.set_xticklabels([f'{sr:.0%}' for sr in suppression_rates])
ax1.set_xlabel('Suppression Rate (at failing site)')
ax1.set_ylabel('Detection Rate (within 150 days)')
ax1.set_title('Can Entropy Monitoring Detect It?', fontweight='bold')
ax1.set_ylim(0, 1.1)
ax1.axhline(1.0, color='gray', linestyle=':', alpha=0.3)
ax1.grid(True, alpha=0.15, axis='y')

# Right: Detection speed
valid_rates = [sr for sr in suppression_rates if results[sr]['mean'] is not None]
means = [results[sr]['mean'] for sr in valid_rates]
stds = [results[sr]['std'] for sr in valid_rates]

ax2.errorbar(range(len(valid_rates)), means, yerr=stds, 
             fmt='o-', color='#E84855', capsize=4, markersize=6, linewidth=1.5)
ax2.axhline(120, color='gray', linestyle='--', alpha=0.5, linewidth=1.5,
            label='Quarterly review (120 days after onset)')
ax2.set_xticks(range(len(valid_rates)))
ax2.set_xticklabels([f'{sr:.0%}' for sr in valid_rates])
ax2.set_xlabel('Suppression Rate (at failing site)')
ax2.set_ylabel('Days After Onset to Detection')
ax2.set_title('How Quickly?', fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.15)

plt.tight_layout()
plt.savefig('entropy_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n--- Summary ---")
print("Even at low suppression rates (10-20%), entropy monitoring usually detects")
print("the failure — but it takes longer and with higher variance.")
print("At 30%+, detection is fast and reliable.")
print("At all rates tested, it outperforms the quarterly review baseline (120 days).")
print("\nCritical caveat: this models 1 failing site out of 50.")
print("For sporadic or very low-magnitude failures, per-site tracking (Figure 4)")
print("is more powerful than aggregate monitoring.")